# Alertas e custos ilustrativos — Extra Trees final — FD001

Será avaliado o primeiro alerta após três ciclos consecutivos com RUL previsto abaixo do limiar. Será considerado oportuno o alerta emitido entre 5 e 30 ciclos antes da falha. A antecedência será contada no ciclo de confirmação, quando a decisão pode efetivamente ser tomada.

Serão utilizadas as previsões OOF por motor do modelo congelado. Como esses folds também foram utilizados durante a otimização, esta etapa será apresentada como análise de desenvolvimento, sem garantia de desempenho independente. O teste oficial contém trajetórias truncadas e não permite avaliar a política completa até a falha. Os horizontes e custos são ilustrativos e não representam requisitos industriais validados.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'cmapss_final_alerts.py').exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Projeto não encontrado')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from cmapss_final_alerts import evaluate_final_alerts
REPORT = ROOT / 'outputs' / 'reports'
predictions = pd.read_csv(REPORT / 'extra-trees-oof-erros-fd001.csv')
events, summary = evaluate_final_alerts(predictions)
events.to_csv(REPORT / 'extra-trees-alertas-eventos-fd001.csv', index=False)
summary.to_csv(REPORT / 'extra-trees-alertas-resumo-fd001.csv', index=False)
display(summary)

resultado,limiar,Prematuro,Oportuno,Tardio,Não detectada,motores
0,10,0,87,13,0,100
1,20,0,100,0,0,100
2,30,15,85,0,0,100
3,40,73,27,0,0,100
4,50,98,2,0,0,100


Insights: Cada motor será contado uma única vez por limiar. Um alerta precoce demais poderá antecipar uma intervenção; um alerta tardio poderá deixar tempo insuficiente para agir. A confirmação reduz a resposta a oscilações isoladas, mas consome parte da antecedência disponível.

In [2]:
order = ['Prematuro', 'Oportuno', 'Tardio', 'Não detectada']
plot = summary.melt(id_vars=['limiar', 'motores'], value_vars=order, var_name='resultado', value_name='quantidade')
plot['percentual'] = 100 * plot.quantidade / plot.motores
fig = px.bar(plot, x='limiar', y='percentual', color='resultado', barmode='stack', category_orders={'resultado': order}, color_discrete_map={'Oportuno': '#2CA02C', 'Prematuro': '#FF7F0E', 'Tardio': '#D62728', 'Não detectada': '#7F7F7F'}, labels={'limiar': 'Limiar de RUL previsto (ciclos)', 'percentual': 'Motores (%)'}, title='Antecedência útil: 5 a 30 ciclos; confirmação: 3 ciclos')
fig.show()
display(events.loc[events.resultado == 'Oportuno'].groupby('limiar').antecedencia.agg(['count', 'median', 'min', 'max']))

,count,median,min,max
limiar,,,,
10,87,7.0,5.0,15.0
20,100,16.0,10.0,26.0
30,85,25.0,16.0,30.0
40,27,28.0,24.0,30.0
50,2,28.5,28.0,29.0


Insights: As barras serão ordenadas pelo limiar crescente. A distribuição da antecedência deverá ser considerada junto com a quantidade de alertas oportunos, pois a mesma classificação pode abranger tempos diferentes para intervenção.

In [3]:
scenarios = {'Intermediário': [3, 1, 10, 10], 'Falha muito cara': [2, 1, 20, 20], 'Intervenção prematura cara': [8, 1, 10, 10]}
rows = []
for scenario, costs in scenarios.items():
    for _, row in summary.iterrows():
        rows.append({'cenario': scenario, 'limiar': row.limiar, 'custo_medio_motor': sum(row[name] * cost for name, cost in zip(order, costs)) / row.motores})
costs = pd.DataFrame(rows)
costs.to_csv(REPORT / 'extra-trees-alertas-custos-fd001.csv', index=False)
display(costs)
px.line(costs, x='limiar', y='custo_medio_motor', color='cenario', markers=True, color_discrete_map={'Intermediário': '#2CA02C', 'Falha muito cara': '#D62728', 'Intervenção prematura cara': '#FF7F0E'}, title='Custos relativos ilustrativos por motor').show()
best = costs.loc[costs.groupby('cenario').custo_medio_motor.idxmin()].sort_values('cenario')
display(best)
text = '# Alertas do Extra Trees final — FD001\n\nAnálise exploratória sobre previsões OOF usadas no desenvolvimento. Janela útil de 5 a 30 ciclos, confirmação em 3 ciclos. Custos relativos ilustrativos; alertas tardios recebem o mesmo custo de falhas não detectadas.\n\n' + '```text\n' + summary.to_string(index=False) + '\n```' + '\n\nMenor custo observado em cada cenário (sem validação independente da escolha):\n\n' + '```text\n' + best.to_string(index=False) + '\n```'
(REPORT / 'alertas-extra-trees-final-fd001.md').write_text(text, encoding='utf-8')

,cenario,limiar,custo_medio_motor
0,Intermediário,10,2.17
1,Intermediário,20,1.00
2,Intermediário,30,1.30
3,Intermediário,40,2.46
4,Intermediário,50,2.96
5,Falha muito cara,10,3.47
6,Falha muito cara,20,1.00
7,Falha muito cara,30,1.15
8,Falha muito cara,40,1.73
9,Falha muito cara,50,1.98


,cenario,limiar,custo_medio_motor
6,Falha muito cara,20,1.0
1,Intermediário,20,1.0
11,Intervenção prematura cara,20,1.0


945

Insights: O menor custo observado será específico das penalidades assumidas. Empates e diferenças pequenas deverão ser preservados na interpretação. A política não será apresentada como validada para implantação, pois custos, horizonte e antecedência mínima precisam ser definidos com responsáveis pela operação. Nesta simulação, o motor continua até a falha mesmo após o alerta; a falha após uma intervenção real não é observada.